In [1]:
import cv2
import mediapipe as mp
import math

print("OpenCV version:", cv2.__version__)
print("MediaPipe version:", mp.__version__)
print("Libraries imported successfully!")

OpenCV version: 4.11.0
MediaPipe version: 0.10.21
Libraries imported successfully!


In [2]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

print("MediaPipe Hands initialized!")

MediaPipe Hands initialized!


In [3]:
def distance(p1, p2):
    return math.sqrt(
        (p1.x - p2.x) ** 2 +
        (p1.y - p2.y) ** 2
    )

print("Distance function created!")

Distance function created!


In [4]:
def recognize_gesture(landmarks):

    # -------------------------
    # LANDMARKS
    # -------------------------

    wrist = landmarks[0]

    thumb_tip = landmarks[4]
    thumb_ip = landmarks[3]

    index_tip = landmarks[8]
    index_pip = landmarks[6]

    middle_tip = landmarks[12]
    middle_pip = landmarks[10]

    ring_tip = landmarks[16]
    ring_pip = landmarks[14]

    pinky_tip = landmarks[20]
    pinky_pip = landmarks[18]

    # -------------------------
    # FINGER DETECTION
    # -------------------------

    index_open = index_tip.y < index_pip.y
    middle_open = middle_tip.y < middle_pip.y
    ring_open = ring_tip.y < ring_pip.y
    pinky_open = pinky_tip.y < pinky_pip.y

    # -------------------------
    # THUMB DETECTION
    # -------------------------

    thumb_up = thumb_tip.y < thumb_ip.y

    # -------------------------
    # THUMBS UP
    # -------------------------

    if (
        thumb_up
        and not index_open
        and not middle_open
        and not ring_open
        and not pinky_open
    ):
        return "Thumbs Up"

    # -------------------------
    # VICTORY / PEACE
    # -------------------------

    if (
        index_open
        and middle_open
        and not ring_open
        and not pinky_open
    ):
        return "Victory / Peace"

    # -------------------------
    # OPEN HAND
    # -------------------------

    if (
        index_open
        and middle_open
        and ring_open
        and pinky_open
    ):
        return "Open Hand"

    # -------------------------
    # FIST
    # -------------------------

    if (
        not index_open
        and not middle_open
        and not ring_open
        and not pinky_open
        and not thumb_up
    ):
        return "Fist"

    return "Unknown"


print("Gesture recognition system ready!")

Gesture recognition system ready!


In [5]:
cap = cv2.VideoCapture(0)

if cap.isOpened():
    print("Webcam opened successfully!")
else:
    print("ERROR: Webcam could not be opened.")

cap.release()

Webcam opened successfully!


In [ ]:
import cv2
import mediapipe as mp

# -------------------------
# MEDIAPIPE SETUP
# -------------------------

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)


# -------------------------
# GESTURE FUNCTION
# -------------------------

def recognize_gesture(landmarks):

    wrist = landmarks[0]

    # Thumb
    thumb_tip = landmarks[4]
    thumb_ip = landmarks[3]

    # Index
    index_tip = landmarks[8]
    index_pip = landmarks[6]

    # Middle
    middle_tip = landmarks[12]
    middle_pip = landmarks[10]

    # Ring
    ring_tip = landmarks[16]
    ring_pip = landmarks[14]

    # Pinky
    pinky_tip = landmarks[20]
    pinky_pip = landmarks[18]

    # Finger states
    index_open = index_tip.y < index_pip.y
    middle_open = middle_tip.y < middle_pip.y
    ring_open = ring_tip.y < ring_pip.y
    pinky_open = pinky_tip.y < pinky_pip.y

    # Thumb
    thumb_up = thumb_tip.y < thumb_ip.y

    # -------------------------
    # THUMBS UP
    # -------------------------

    if (
        thumb_up
        and not index_open
        and not middle_open
        and not ring_open
        and not pinky_open
    ):
        return "Thumbs Up"

    # -------------------------
    # VICTORY / PEACE
    # -------------------------

    if (
        index_open
        and middle_open
        and not ring_open
        and not pinky_open
    ):
        return "Victory / Peace"

    # -------------------------
    # OPEN HAND
    # -------------------------

    if (
        index_open
        and middle_open
        and ring_open
        and pinky_open
    ):
        return "Open Hand"

    # -------------------------
    # FIST
    # -------------------------

    if (
        not index_open
        and not middle_open
        and not ring_open
        and not pinky_open
        and not thumb_up
    ):
        return "Fist"

    return "Unknown"


# -------------------------
# START WEBCAM
# -------------------------

cap = cv2.VideoCapture(0)

if not cap.isOpened():

    print("ERROR: Could not open webcam.")

else:

    print("AI Hand Gesture Recognition Started")
    print("Show your hand to the camera.")
    print("Press Q to exit.")

    while True:

        # Read camera frame
        ret, frame = cap.read()

        if not ret:
            print("Could not read camera frame.")
            break

        # Mirror image
        frame = cv2.flip(frame, 1)

        # Convert BGR to RGB
        rgb_frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        # Process hand
        result = hands.process(rgb_frame)

        gesture = "No Hand Detected"

        # If hand detected
        if result.multi_hand_landmarks:

            for hand_landmarks in result.multi_hand_landmarks:

                # Draw hand landmarks
                mp_drawing.draw_landmarks(
                    frame,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS
                )

                # Recognize gesture
                gesture = recognize_gesture(
                    hand_landmarks.landmark
                )

        # -------------------------
        # DISPLAY GESTURE
        # -------------------------

        cv2.putText(
            frame,
            "Gesture: " + gesture,
            (30, 60),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.1,
            (0, 255, 0),
            3
        )

        cv2.putText(
            frame,
            "Open Hand | Fist | Thumbs Up | Victory",
            (20, 100),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            2
        )

        cv2.putText(
            frame,
            "Press Q to Exit",
            (30, 135),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2
        )

        # Show window
        cv2.imshow(
            "AI Hand Gesture Recognition",
            frame
        )

        # Exit
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    # Release camera
    cap.release()
    cv2.destroyAllWindows()

    print("Program stopped.")

AI Hand Gesture Recognition Started
Show your hand to the camera.
Press Q to exit.


In [ ]:
def recognize_gesture(landmarks):

    # Thumb
    thumb_tip = landmarks[4]
    thumb_ip = landmarks[3]

    # Other fingers
    index_tip = landmarks[8]
    index_pip = landmarks[6]

    middle_tip = landmarks[12]
    middle_pip = landmarks[10]

    ring_tip = landmarks[16]
    ring_pip = landmarks[14]

    pinky_tip = landmarks[20]
    pinky_pip = landmarks[18]

    # Check thumb
    thumb_up = thumb_tip.y < thumb_ip.y

    # Check fingers folded
    index_closed = index_tip.y > index_pip.y
    middle_closed = middle_tip.y > middle_pip.y
    ring_closed = ring_tip.y > ring_pip.y
    pinky_closed = pinky_tip.y > pinky_pip.y

    # THUMBS UP
    if thumb_up and index_closed and middle_closed and ring_closed and pinky_closed:
        return "Thumbs Up"

    # VICTORY / PEACE
    if (index_tip.y < index_pip.y and
        middle_tip.y < middle_pip.y and
        ring_tip.y > ring_pip.y and
        pinky_tip.y > pinky_pip.y):
        return "Victory / Peace"

    # OPEN HAND
    if (index_tip.y < index_pip.y and
        middle_tip.y < middle_pip.y and
        ring_tip.y < ring_pip.y and
        pinky_tip.y < pinky_pip.y):
        return "Open Hand"

    # FIST
    if (index_closed and middle_closed and ring_closed and pinky_closed):
        return "Fist"

    return "Unknown"


print("Gesture function updated!")

In [ ]:
def recognize_gesture(landmarks):

    # Coordinates
    wrist = landmarks[0]

    thumb_tip = landmarks[4]
    index_tip = landmarks[8]
    middle_tip = landmarks[12]
    ring_tip = landmarks[16]
    pinky_tip = landmarks[20]

    index_pip = landmarks[6]
    middle_pip = landmarks[10]
    ring_pip = landmarks[14]
    pinky_pip = landmarks[18]

    # -------------------------
    # Finger states
    # -------------------------
    index_open = index_tip.y < index_pip.y
    middle_open = middle_tip.y < middle_pip.y
    ring_open = ring_tip.y < ring_pip.y
    pinky_open = pinky_tip.y < pinky_pip.y

    # -------------------------
    # THUMB UP
    # -------------------------
    # Your thumb tip is above the thumb IP
    thumb_up = thumb_tip.y < landmarks[3].y

    # If thumb is clearly above the other fingers,
    # treat it as thumbs up.
    thumb_above_fingers = (
        thumb_tip.y < index_tip.y and
        thumb_tip.y < middle_tip.y and
        thumb_tip.y < ring_tip.y and
        thumb_tip.y < pinky_tip.y
    )

    if thumb_up and thumb_above_fingers:
        return "Thumbs Up"

    # -------------------------
    # VICTORY / PEACE
    # -------------------------
    if (
        index_open and
        middle_open and
        not ring_open and
        not pinky_open
    ):
        return "Victory / Peace"

    # -------------------------
    # OPEN HAND
    # -------------------------
    if (
        index_open and
        middle_open and
        ring_open and
        pinky_open
    ):
        return "Open Hand"

    # -------------------------
    # FIST
    # -------------------------
    if (
        not index_open and
        not middle_open and
        not ring_open and
        not pinky_open
    ):
        return "Fist"

    return "Unknown"


print("Gesture function updated successfully!")